## 📊 Task 2: Free vs Paid Apps Analysis
**Author:** Vansh Sharma


- Cleaned and transformed dataset columns required for analysis.
- Converted Price into numeric format and created Revenue column.
- Applied filters:
  - Android Version > 4.0
  - Size > 15 MB
  - Installs >= 10,000
  - Revenue >= $10,000
  - Content Rating = Everyone
  - App name length <= 30 characters
- Identified top 3 app categories based on total installs.
- Compared average installs and revenue for Free vs Paid apps using a dual-axis bar chart.
- Added time-based restriction: graph is displayed only between **1 PM IST to 2 PM IST**.


# Import Libraries


In [84]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime
import pytz
import matplotlib.pyplot as plt
from zoneinfo import ZoneInfo


 

# Load DataSet `play_store_data`

In [89]:
play_store_data = pd.read_csv(
    r"C:\Users\Vansh Sharma\Downloads\Play Store Data (1).csv"
)



In [86]:
play_store_data.head(3)

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up


In [95]:
play_store_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10841 entries, 0 to 10840
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             10841 non-null  object 
 1   Category        10841 non-null  object 
 2   Rating          9367 non-null   float64
 3   Reviews         10841 non-null  object 
 4   Size            9146 non-null   object 
 5   Installs        10841 non-null  object 
 6   Type            10840 non-null  object 
 7   Price           10841 non-null  object 
 8   Content Rating  10840 non-null  object 
 9   Genres          10841 non-null  object 
 10  Last Updated    10841 non-null  object 
 11  Current Ver     10833 non-null  object 
 12  Android Ver     10838 non-null  object 
dtypes: float64(1), object(12)
memory usage: 1.1+ MB


## Size Conversion

- Converted app size values into MB format.
- Converted KB values into MB.
- Removed non-numeric size values.
- Changed Size column into numeric format for filtering.

In [96]:
# Convert Size into MB

# Replace non-size values
play_store_data["Size"] = (
    play_store_data["Size"]
    .replace("Varies with device", np.nan)
)


# Remove unwanted characters
play_store_data["Size"] = (
    play_store_data["Size"]
    .astype(str)
    .str.replace(",", "", regex=False)
)


# Convert KB to MB
mask_k = play_store_data["Size"].str.contains("k", na=False)

play_store_data.loc[mask_k, "Size"] = (
    play_store_data.loc[mask_k, "Size"]
    .str.replace("k", "", regex=False)
    .astype(float)
    / 1024
)


# Convert MB values
mask_m = play_store_data["Size"].str.contains("M", na=False)

play_store_data.loc[mask_m, "Size"] = (
    play_store_data.loc[mask_m, "Size"]
    .str.replace("M", "", regex=False)
    .astype(float)
)


# Convert remaining invalid values to NaN
play_store_data["Size"] = pd.to_numeric(
    play_store_data["Size"],
    errors="coerce"
)


# Fill missing size values
play_store_data["Size"] = play_store_data["Size"].fillna(
    play_store_data["Size"].median()
)


# Check dtype
play_store_data["Size"].dtype

dtype('float64')

### Data Cleaning

- Removed corrupted rows.
- Converted Installs and Reviews into numeric format.
- Converted Last Updated into datetime format.
- Handled missing values using forward fill and mode imputation.

In [97]:
# Remove corrupted row
play_store_data = play_store_data[
    play_store_data["Installs"] != "Free"
]

# Remove commas and plus signs + convert to integer
play_store_data["Installs"] = (
    play_store_data["Installs"]
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False)
    .astype(int)
)

# Convert Last Updated to datetime
play_store_data["Last Updated"] = pd.to_datetime(
    play_store_data["Last Updated"],
    errors="coerce"
)

# Convert Reviews into integer
play_store_data["Reviews"] = play_store_data["Reviews"].astype(int)

# Fill missing values
play_store_data["Rating"] = play_store_data["Rating"].ffill()

play_store_data["Current Ver"] = play_store_data["Current Ver"].ffill()

play_store_data["Android Ver"] = play_store_data["Android Ver"].ffill()

play_store_data["Type"] = play_store_data["Type"].fillna(
    play_store_data["Type"].mode()[0]
)

In [98]:
# Rmove Everyone "garbage"
play_store_data = play_store_data[
    play_store_data["Price"] != "Everyone"
]

# Convert Price column from object to float

play_store_data["Price"] = (
    play_store_data["Price"]
    .replace("Free", "0")                 # Replace Free apps price with 0
    .str.replace("$", "", regex=False)    # Remove dollar sign
    .astype(float)                       # Convert string to float
)


# Check datatype
print(play_store_data["Price"].dtype)

float64


### Revenue Calculation

- Created a new **Revenue** column by multiplying app price with total installs.
- Used this metric to compare revenue performance between Free and Paid apps.

In [99]:
play_store_data["Revenue"] = (
    play_store_data["Price"] *
    play_store_data["Installs"]
)

### Android Version Filter

- Converted Android version into a numeric format.
- Filtered apps with Android version greater than **4.0** as per requirement.

In [100]:
#Convert Android Version into numeric format

play_store_data["Android Ver"] = (
    play_store_data["Android Ver"]
    .str.extract(r"(\d+\.\d+)")
    .astype(float)
)


#Filter Android version greater than 4.0

play_store_data = play_store_data[
    play_store_data["Android Ver"] > 4.0
]


# Check result

play_store_data[["App", "Android Ver"]].head()

,App,Android Ver
3,Sketch - Draw & Paint,4.2
4,Pixel Draw - Number Art Coloring Book,4.4
7,Infinite Painter,4.2
10,Text on Photo - Fonteee,4.1
12,Tattoo Name On My Photo Editor,4.1


### Content Rating & App Length Filter

- Filtered apps with **Content Rating = Everyone**.
- Removed apps having names longer than **30 characters** (including spaces and special characters).

In [101]:
# Content Rating Filter
play_store_data = play_store_data[
    play_store_data["Content Rating"] == "Everyone"
]


# App Name Length Filter
play_store_data["App_Length"] = (
    play_store_data["App"].str.len()
)

play_store_data = play_store_data[
    play_store_data["App_Length"] <= 30
]


# Check result
play_store_data.shape

(3123, 15)

### Size Filter

- Filtered applications with size greater than **15 MB**.
- Kept only larger apps for further revenue and install analysis.

In [102]:
# Filter apps with size greater than 15 MB

play_store_data = play_store_data[
    play_store_data["Size"] > 15
]


# Check result
play_store_data.shape

(1488, 15)

### Installs & Revenue Filter

- Filtered apps with **minimum 10,000 installs**.
- Removed apps with **revenue below $10,000**.
- Prepared high-performing apps for further analysis.

In [103]:
play_store_data = play_store_data[
    (play_store_data["Installs"] >= 10000) &
    (play_store_data["Revenue"] >= 10000)
]


# Check shape
play_store_data.shape

(33, 15)





### Top 3 Categories Identification

- Grouped apps category-wise and calculated total installs.
- Sorted categories based on highest number of installs.
- Selected the top 3 performing app categories for further Free vs Paid analysis.

In [104]:
top_categories = (
    play_store_data
    .groupby("Category", as_index=False)
    ["Installs"]
    .sum()
    .sort_values(
        "Installs",
        ascending=False
    )
    .head(3)
)

top_categories

,Category,Installs
8,PHOTOGRAPHY,3000000
1,FAMILY,2670000
7,PERSONALIZATION,2010000


### Top Category Data Filtering

- Filtered the dataset to keep only apps belonging to the identified top 3 categories.
- Used these categories for Free vs Paid app performance comparison.

In [105]:
play_store_data = play_store_data[
    play_store_data["Category"].isin(
        top_categories["Category"]
    )
]

### Free vs Paid App Comparison

- Grouped apps based on their type (**Free** and **Paid**).
- Calculated average installs and average revenue for each app type.
- Rounded values for better readability before visualization.

In [ ]:
comparison = (
    play_store_data
    .groupby("Type", as_index=False)
    .agg({
        "Installs": "mean",
        "Revenue": "mean"
    })
)

comparison["Installs"] = comparison["Installs"].round(0)
comparison["Revenue"] = comparison["Revenue"].round(2)

comparison

,Type,Installs,Revenue
0,Paid,590769.0,1965246.15


### Dual Axis Chart with Time Restriction

- Created a dual-axis bar chart to compare **Average Installs** and **Average Revenue** for Free vs Paid apps.
- Added IST time-based access control.
- Graph is displayed only between **1 PM IST and 2 PM IST**.

In [108]:
# Current IST Time
ist = pytz.timezone("Asia/Kolkata")
current_time = datetime.now(ist).time()


# Allowed time range
start_time = datetime.strptime("13:00", "%H:%M").time()
end_time = datetime.strptime("14:00", "%H:%M").time()


# Show graph only between 1 PM - 2 PM IST

if start_time <= current_time <= end_time:

    x = np.arange(len(comparison["Type"]))
    width = 0.35


    fig, ax1 = plt.subplots(figsize=(10,6))


    # Average Installs
    ax1.bar(
        x - width/2,
        comparison["Installs"],
        width,
        label="Average Installs"
    )

    ax1.set_xlabel("App Type")
    ax1.set_ylabel("Average Installs")

    ax1.set_xticks(x)
    ax1.set_xticklabels(
        comparison["Type"]
    )


    # Average Revenue
    ax2 = ax1.twinx()

    ax2.bar(
        x + width/2,
        comparison["Revenue"],
        width,
        label="Average Revenue"
    )

    ax2.set_ylabel(
        "Average Revenue ($)"
    )


    plt.title(
        "Average Installs vs Revenue: Free vs Paid Apps"
    )


    fig.legend(
        loc="upper right",
        bbox_to_anchor=(0.9,0.9)
    )


    plt.grid(axis="y")
    plt.show()


else:
    print(
        "Graph is available only between 1 PM IST and 2 PM IST"
    )

Graph is available only between 1 PM IST and 2 PM IST


# `KPIs`

In [109]:
# Total Apps
total_apps = play_store_data["App"].nunique()

# Total Categories
total_categories = play_store_data["Category"].nunique()

# Best Performing Category
top_category = (
    play_store_data.groupby("Category")["Installs"]
    .sum()
    .idxmax()
)

# Average Installs
avg_installs = (
    play_store_data["Installs"]
    .mean()
    .round(0)
)

# Average Revenue
avg_revenue = (
    play_store_data["Revenue"]
    .mean()
    .round(2)
)

# Highest Revenue App Type
highest_revenue_type = (
    play_store_data.groupby("Type")["Revenue"]
    .mean()
    .idxmax()
)


print("Total Apps:", total_apps)
print("Total Categories:", total_categories)
print("Top Category by Installs:", top_category)
print("Average Installs:", avg_installs)
print("Average Revenue: $", avg_revenue)
print("Highest Revenue App Type:", highest_revenue_type)

Total Apps: 10
Total Categories: 3
Top Category by Installs: PHOTOGRAPHY
Average Installs: 590769.0
Average Revenue: $ 1965246.15
Highest Revenue App Type: Paid


### Key Performance Indicators (KPIs)

- **Total Apps Analyzed:** 10

- **Total Categories Covered:** 3

- **Top Performing Category:** Photography

- **Average Installs per App:** 590.77K

- **Average Revenue per App:** $1.96M

- **Highest Revenue Generating Type:** Paid Apps